In [ ]:
import os
import json
import random
import cv2
import numpy as np
import math
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import urllib.request
import urllib.error
import nltk
import re
from datasets import load_dataset
from tqdm.notebook import tqdm
import zipfile
import glob

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# Шрифт для генерации

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-roboto fonts-ubuntu fonts-pt fonts-liberation fonts-noto > /dev/null

system_fonts_dir = "/usr/share/fonts/truetype/"
font_paths = glob.glob(os.path.join(system_fonts_dir, "**/*.ttf"), recursive=True)

if not font_paths:
    print("системные шрифты не найдены, надо на линуксе запускать (или колаб)")
else:
    print(f"найдено: {len(font_paths)} шрифтов")

    print("примеры путей:")
    for p in font_paths[:5]:
        print(f"{p}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package fonts-pt
найдено: 17 шрифтов
примеры путей:
/usr/share/fonts/truetype/humor-sans/Humor-Sans.ttf
/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf
/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf
/usr/share/fonts/truetype/liberation/LiberationMono-Italic.ttf
/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf


# Генерация текстов

In [ ]:
def build_ocr_text_corpus(output_filename="ocr_corpus.txt", lines_per_source=50000):
    corpus_sentences = set()

    print("1) русская вики")
    wiki_ru = load_dataset("wikimedia/wikipedia", "20231101.ru", split="train", streaming=True)
    for row in tqdm(wiki_ru, total=lines_per_source):
        sentences = nltk.sent_tokenize(row['text'], language="russian")
        corpus_sentences.update([s.strip() for s in sentences if 10 < len(s) < 150])
        if len(corpus_sentences) >= lines_per_source:
            break

    print("2) англ вики")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    for row in tqdm(wiki_en, total=lines_per_source):
        sentences = nltk.sent_tokenize(row['text'], language="english")
        corpus_sentences.update([s.strip() for s in sentences if 10 < len(s) < 150])
        if len(corpus_sentences) >= lines_per_source * 2:
            break

    print("3) статьи с хабра (ру/англ)")
    habr = load_dataset("IlyaGusev/habr", split="train", streaming=True)
    for row in tqdm(habr, total=lines_per_source):
        text = str(row['text_markdown'])
        sentences = nltk.sent_tokenize(text, language="russian")
        valid_sentences = [s.strip() for s in sentences if 10 < len(s) < 150 and re.search('[a-zA-Zа-яА-Я]', s)]
        corpus_sentences.update(valid_sentences)
        if len(corpus_sentences) >= lines_per_source * 3:
            break

    final_corpus = list(corpus_sentences)
    random.shuffle(final_corpus)

    with open(output_filename, "w", encoding="utf-8") as f:
        for line in final_corpus:
            f.write(line + "\n")

    print(f"реди, всего строк {len(final_corpus)}")
    return final_corpus

In [ ]:
corpus_list = build_ocr_text_corpus("ocr_corpus.txt", lines_per_source=30000)

1) русская вики


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/30000 [00:00<?, ?it/s]

...

3) статьи с хабра (ру/англ)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

  0%|          | 0/30000 [00:00<?, ?it/s]

реди, всего строк 90055


# Генерация картинок

In [ ]:
def generate_textured_background(width, height):
    """Генерирует сложный фон (градиент + текстурный шум)"""
    c1 = np.random.randint(150, 256, (3,), dtype=np.uint8)
    c2 = np.random.randint(50, 180, (3,), dtype=np.uint8)
    bg = np.zeros((height, width, 3), dtype=np.uint8)
    for y in range(height):
        ratio = y / height
        bg[y, :] = (c1 * (1 - ratio) + c2 * ratio).astype(np.uint8)
    noise = np.random.normal(0, 15, (height, width, 3)).astype(np.int16)
    bg = cv2.add(bg.astype(np.int16), noise)
    return Image.fromarray(np.clip(bg, 0, 255).astype(np.uint8))

In [ ]:
def get_rotated_points(x, y, w, h, angle_deg, pad):
    """Вычисляет 4 точки bounding box (границы текста) после поворота"""
    points = np.array([
        [-w/2 - pad, -h/2 - pad],
        [w/2 + pad, -h/2 - pad],
        [w/2 + pad, h/2 + pad],
        [-w/2 - pad, h/2 + pad]
    ])
    angle_rad = np.radians(angle_deg)
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array(((c, -s), (s, c)))
    rotated_points = points.dot(R.T)
    final_points = rotated_points + [x + w/2, y + h/2]
    return final_points.astype(int).tolist()

In [ ]:
def generate_dataset(corpus, fonts, num_images=10, output_dir="dataset", negative_ratio=0.10):
    """Генерирует датасет для обучения и валидации """
    os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
    annotation_file = open(os.path.join(output_dir, "train_labels.txt"), "w", encoding="utf-8")

    num_negatives = int(math.ceil(num_images * negative_ratio))
    is_empty_list = [True] * num_negatives + [False] * (num_images - num_negatives)
    random.shuffle(is_empty_list)

    print(f"всего фото: {num_images} из них с текстом: {num_images - num_negatives}")

    for i, is_empty in enumerate(is_empty_list):
        bg_width, bg_height = 800, 800
        main_img = generate_textured_background(bg_width, bg_height)
        labels = []

        if not is_empty:
            current_y = random.randint(30, 80)
            num_lines = random.randint(1, 8)
            lines_drawn = 0

            for _ in range(num_lines):
                text = random.choice(corpus)
                current_font_path = random.choice(fonts)
                f_size = random.randint(22, 45)

                try:
                    font = ImageFont.truetype(current_font_path, size=f_size)
                except:
                    continue

                dummy_draw = ImageDraw.Draw(Image.new("RGBA", (1, 1)))
                words = text.split()
                valid_text, tw, th = "", 0, 0

                while words:
                    candidate = " ".join(words)
                    bbox = dummy_draw.textbbox((0, 0), candidate, font=font)
                    tw = bbox[2] - bbox[0]
                    th = (bbox[3] - bbox[1]) + 10
                    if tw <= bg_width - 120:
                        valid_text = candidate
                        break
                    words.pop()

                if not valid_text:
                    continue

                text_layer = Image.new("RGBA", (tw, th + 10), (255, 255, 255, 0))
                text_draw = ImageDraw.Draw(text_layer)
                t_color = (random.randint(0, 40), random.randint(0, 40), random.randint(0, 40), 255)
                text_draw.text((0, 5), valid_text, fill=t_color, font=font)

                angle = random.uniform(-3, 3)
                rotated_layer = text_layer.rotate(angle, expand=True, resample=Image.Resampling.BICUBIC)
                new_tw, new_th = rotated_layer.size

                start_x = random.randint(40, max(41, bg_width - new_tw - 40))
                main_img.paste(rotated_layer, (start_x, current_y), rotated_layer)

                pts = get_rotated_points(start_x, current_y, new_tw, new_th, angle, pad=2)
                labels.append({"transcription": valid_text, "points": pts})

                current_y += new_th + random.randint(15, 35)
                lines_drawn += 1
                if current_y > bg_height - 60:
                    break

            if lines_drawn == 0:
                font = ImageFont.truetype(fonts[0], size=30)
                fallback = f"ID: {random.randint(1000, 9999)}"
                bbox = dummy_draw.textbbox((0, 0), fallback, font=font)
                tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1] + 10
                t_layer = Image.new("RGBA", (tw, th + 10), (255, 255, 255, 0))
                ImageDraw.Draw(t_layer).text((0, 5), fallback, fill=(0,0,0,255), font=font)
                main_img.paste(t_layer, (50, 50), t_layer)
                pts = get_rotated_points(50, 50, tw, th, 0, pad=2)
                labels.append({"transcription": fallback, "points": pts})

        img_filename = f"image_{i:04d}.jpg"
        main_img.convert("RGB").save(os.path.join(output_dir, "images", img_filename))

        anno_str = json.dumps(labels, ensure_ascii=False)
        annotation_file.write(f"images/{img_filename}\t{anno_str}\n")

    annotation_file.close()
    print("датасет сгенерирован")

In [ ]:

generate_dataset(corpus_list, font_paths, num_images=50, negative_ratio=0.10)

всего фото: 50 из них с текстом: 45
датасет сгенерирован


# Обучение

In [ ]:
!python -m pip install paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple
# долгий шаг
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

!pip install -r requirements.txt
!pip install pyclipper shapely

Looking in indexes: https://mirror.baidu.com/pypi/simple
ERROR: Could not find a version that satisfies the requirement paddlepaddle-gpu (from versions: none)
ERROR: No matching distribution found for paddlepaddle-gpu
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 339143, done.
remote: Counting objects: 100% (1547/1547), done.
remote: Compressing objects: 100% (439/439), done.
remote: Total 339143 (delta 1304), reused 1108 (delta 1108), pack-reused 337596 (from 3)
Receiving objects: 100% (339143/339143), 1.80 GiB | 22.93 MiB/s, done.
Resolving deltas: 100% (268762/268762), done.
Updating files: 100% (2220/2220), done.
/content/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00


In [ ]:
!pip install paddlepaddle-gpu
%cd /content/PaddleOCR

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 692.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


/content/PaddleOCR


In [ ]:
dataset_path = "../dataset"
labels_file = os.path.join(dataset_path, "train_labels.txt")

with open(labels_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()

random.shuffle(lines)
split_idx = int(len(lines) * 0.8)
train_lines = lines[:split_idx]
test_lines = lines[split_idx:]

with open(os.path.join(dataset_path, "train.txt"), 'w', encoding='utf-8') as f:
    f.writelines(train_lines)
with open(os.path.join(dataset_path, "test.txt"), 'w', encoding='utf-8') as f:
    f.writelines(test_lines)

print(f"Train: {len(train_lines)} Test: {len(test_lines)}")

Train: 40 Test: 10


# бейзлайн

In [ ]:
config_dir = "configs/det/ch_PP-OCRv4"
os.makedirs(config_dir, exist_ok=True)

config_text = """Global:
  debug: false
  use_gpu: true
  epoch_num: 50
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/ch_PP-OCRv4_det_student
  save_epoch_step: 10
  eval_batch_step: [0, 50]
  cal_metric_during_train: False
  pretrained_model: ./pretrain_models/ch_PP-OCRv4_det_train/student
  checkpoints:
  save_inference_dir: null
  use_visualdl: false
  infer_img: doc/imgs_en/img_10.jpg
  save_res_path: ./output/det_db/predicts_db.txt

Architecture:
  model_type: det
  algorithm: DB
  Transform:
  Backbone:
    name: PPLCNetV3
    scale: 0.75
    det: true
  Neck:
    name: RSEFPN
    out_channels: 96
    shortcut: true
  Head:
    name: DBHead
    k: 50

Loss:
  name: DBLoss
  balance_loss: true
  main_loss_type: DiceLoss
  alpha: 5
  beta: 10
  ohem_ratio: 3

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2
  regularizer:
    name: 'L2'
    factor: 5.0e-05

PostProcess:
  name: DBPostProcess
  thresh: 0.3
  box_thresh: 0.6
  max_candidates: 1000
  unclip_ratio: 1.5

Metric:
  name: DetMetric
  main_indicator: hmean

Train:
  dataset:
    name: SimpleDataSet
    data_dir: ../dataset
    label_file_list:
      - ../dataset/train.txt
    ratio_list: [1.0]
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - DetLabelEncode: null
      - IaaAugment:
          augmenter_args:
            - { 'type': Fliplr, 'args': { 'p': 0.5 } }
            - { 'type': Affine, 'args': { 'rotate': [-10, 10] } }
            - { 'type': Resize, 'args': { 'size': [0.5, 3] } }
      - EastRandomCropData:
          size: [640, 640]
          max_tries: 50
          keep_ratio: true
      - MakeBorderMap:
          shrink_ratio: 0.4
          thresh_min: 0.3
          thresh_max: 0.7
      - MakeShrinkMap:
          shrink_ratio: 0.4
          min_text_size: 8
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage: null
      - KeepKeys:
          keep_keys: ['image', 'threshold_map', 'threshold_mask', 'shrink_map', 'shrink_mask']
  loader:
    shuffle: true
    drop_last: false
    batch_size_per_card: 8
    num_workers: 4

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ../dataset
    label_file_list:
      - ../dataset/test.txt
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - DetLabelEncode: null
      - DetResizeForTest:
          limit_side_len: 960
          limit_type: max
      - NormalizeImage:
          scale: 1./255.
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]
          order: 'hwc'
      - ToCHWImage: null
      - KeepKeys:
          keep_keys: ['image', 'shape', 'polys', 'ignore_tags']
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 1
    num_workers: 2
"""

file_path = os.path.join(config_dir, "ch_PP-OCRv4_det_student.yml")
with open(file_path, "w", encoding="utf-8") as f:
    f.write(config_text)

print(f"конфиг сохранен по пути: {file_path}")

конфиг сохранен по пути: configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml


In [ ]:
!python tools/eval.py -c configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml \
    -o Global.pretrained_model="./pretrain_models/ch_PP-OCRv4_det_train/best_accuracy" \
    Eval.dataset.data_dir="../dataset" \
    Eval.dataset.label_file_list=["../dataset/test.txt"]

/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(
Skipping import of the encryption module.
E0511 18:54:42.983317 18088 place.cc:333] Cannot use GPU because there is no GPU detected on your machine.


# тут должен быть eda анализ

## резы
INFO: precision:0.25

pocr INFO: recall:0.25

INFO: hmean:0.25

# дообучение модели

In [ ]:
!python tools/train.py -c configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml \
    -o Global.pretrained_model="./pretrain_models/ch_PP-OCRv4_det_train/best_accuracy" \
    Global.epoch_num=50 \
    Global.eval_batch_step='[0, 50]' \
    Global.save_epoch_step=10 \
    Train.dataset.data_dir="../dataset" \
    Train.dataset.label_file_list=["../dataset/train.txt"] \
    Train.loader.batch_size_per_card=8 \
    Eval.dataset.data_dir="../dataset" \
    Eval.dataset.label_file_list=["../dataset/test.txt"]

/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(
Skipping import of the encryption module.
E0511 18:54:53.751199 18184 place.cc:333] Cannot use GPU because there is no GPU detected on your machine.


In [ ]:
!python tools/eval.py -c configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml \
    -o Global.pretrained_model="./output/ch_PP-OCRv4_det_student/best_accuracy" \
    Eval.dataset.data_dir="../dataset" \
    Eval.dataset.label_file_list=["../dataset/test.txt"]

/usr/local/lib/python3.12/dist-packages/paddle/base/framework.py:688: UserWarning: You are using GPU version Paddle, but your CUDA device is not set properly. CPU device will be used by default.
  warnings.warn(
Skipping import of the encryption module.
E0511 18:55:02.322221 18256 place.cc:333] Cannot use GPU because there is no GPU detected on your machine.


## на 50 картинках
ppocr INFO: precision:0.9459459459459459

ppocr INFO: recall:0.6730769230769231

ppocr INFO: hmean:0.7865168539325843

улучшило)